# Demo 06 : Conflicts

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


In [2]:
import json
import operator
import sqlite3
import threading
import time
from pathlib import Path
from typing import Annotated, Literal
from typing_extensions import TypedDict

from langchain.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.callbacks import BaseCallbackHandler
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command


class Counter(BaseCallbackHandler):
    def __init__(self):
        self.lock = threading.Lock()
        self.calls = self.input_tokens = self.output_tokens = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        with self.lock:
            self.calls += 1

    def on_llm_end(self, response, **kwargs):
        with self.lock:
            for batch in response.generations:
                for generation in batch:
                    usage = generation.message.usage_metadata or {}
                    self.input_tokens += usage.get("input_tokens", 0)
                    self.output_tokens += usage.get("output_tokens", 0)


counter = Counter()
llm = make_llm(callbacks=[counter], max_tokens=500, max_retries=0)
started = time.perf_counter()
DB = Path(data("demo-world/service.db")).resolve()
MANUALS = Path(data_folder("demo-world/manuals"))
NOTEBOOK_DIR = ROOT / "notebooks" / "06-agent-communication" if ROOT else Path.cwd()


def stock(part="CE-CF-02"):
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        db.row_factory = sqlite3.Row
        row = db.execute("SELECT part_no, name, stock, price_eur, depot FROM parts WHERE part_no = ?", (part,)).fetchone()
    return dict(row) if row else {"part_no": part, "error": "unknown part"}


def guidance(error="F-CE-110"):
    return "\n".join(f"[{p.name}] {line}" for p in sorted(MANUALS.glob("*.md"))
                     for line in p.read_text(encoding="utf-8").splitlines()
                     if line.startswith("| " + error + " |"))


FACTS = stock()
QUESTION = "Report CE-CF-02 stock and location, then the first action and replacement condition for F-CE-110."
LIMIT = {"recursion_limit": 16}


def finish_metrics(name):
    row = {"model": MODEL, "rps": float(os.environ.get("COURSE_RPS", "2")),
           "calls": counter.calls, "input_tokens": counter.input_tokens,
           "output_tokens": counter.output_tokens, "seconds": round(time.perf_counter() - started, 1)}
    print("metrics :", row)
    (NOTEBOOK_DIR / (name + "-metrics.json")).write_text(json.dumps(row, indent=2) + "\n", encoding="utf-8")

## a · conflicting-writes

We give seller and buyer opposite proposals for the same stock. Two concurrent writes to an unreduced decision key raise InvalidUpdateError. Keeping both proposals makes the conflict inspectable.

In [3]:
from langgraph.errors import InvalidUpdateError


class DecisionState(TypedDict):
    decision: str


bad = StateGraph(DecisionState)
bad.add_node("seller", lambda s: {"decision": "sell CE-CF-02"})
bad.add_node("buyer", lambda s: {"decision": "retain CE-CF-02"})
for role in ["seller", "buyer"]:
    bad.add_edge(START, role)
    bad.add_edge(role, END)
try:
    bad.compile().invoke({"decision": ""})
except InvalidUpdateError as exc:
    print("conflict :", str(exc).splitlines()[0])


class Proposals(TypedDict):
    proposals: Annotated[list[dict], operator.add]


fixed = StateGraph(Proposals)
fixed.add_node("seller", lambda s: {"proposals": [{"role": "seller", "action": "sell"}]})
fixed.add_node("buyer", lambda s: {"proposals": [{"role": "buyer", "action": "retain"}]})
for role in ["seller", "buyer"]:
    fixed.add_edge(START, role)
    fixed.add_edge(role, END)
print("retained proposals :", fixed.compile().invoke({"proposals": []}))

conflict : At key 'decision': Can receive only one value per step. Use an Annotated key to handle multiple values.
retained proposals : {'proposals': [{'role': 'buyer', 'action': 'retain'}, {'role': 'seller', 'action': 'sell'}]}


## b · voting

We collect independent votes about selling cooling-fan stock. The reserve is a made-up policy input, not a field in the inventory database. We retain the evidence and reasons alongside each vote.

In [4]:
from typing import Literal
from pydantic import BaseModel


class Vote(BaseModel):
    action: Literal["sell", "retain"]
    reason: str


class VoteState(TypedDict):
    facts: dict
    reserve: int
    votes: Annotated[list[dict], operator.add]


def voter(role):
    def node(state):
        vote = llm.with_structured_output(Vote).invoke(
            f"Assess the proposal from the {role} role. Policy: never reduce stock below reserve. "
            "Proposal: sell four CE-CF-02.\n" + json.dumps({"facts": state["facts"], "reserve": state["reserve"]}))
        return {"votes": [{"role": role, **vote.model_dump()}]}
    return node


voting = StateGraph(VoteState)
for role in ["inventory", "workshop", "procurement"]:
    voting.add_node(role, voter(role))
    voting.add_edge(START, role)
    voting.add_edge(role, END)
votes = voting.compile().invoke({"facts": FACTS, "reserve": 4, "votes": []})["votes"]
counts = {action: sum(v["action"] == action for v in votes) for action in ["sell", "retain"]}
for vote in sorted(votes, key=lambda v: v["role"]):
    print("vote :", vote)
print("counts :", counts)
print("majority :", max(counts, key=counts.get))

vote : {'role': 'inventory', 'action': 'retain', 'reason': 'Selling four would reduce stock from 6 to 2, below the reserve of 4.'}
vote : {'role': 'procurement', 'action': 'retain', 'reason': 'Selling four would reduce stock from 6 to 2, below the reserve of 4.'}
vote : {'role': 'workshop', 'action': 'retain', 'reason': 'Selling four would reduce stock from 6 to 2, below the reserve of 4.'}
counts : {'sell': 0, 'retain': 3}
majority : retain


## c · judge-and-tie-break

We supply a deliberately tied pair to make the tie rule visible. A model judge explains the proposals, while code applies the reserve invariant and a fixed retain-on-tie rule.

In [5]:
tied = [{"role": "seller", "action": "sell", "reason": "Release storage."},
        {"role": "buyer", "action": "retain", "reason": "Keep workshop reserve."}]
judge = llm.with_structured_output(Vote).invoke(
    "Judge these proposals against the evidence and the reserve. The proposal sells four units.\n"
    + json.dumps({"facts": FACTS, "reserve": 4, "proposals": tied}))


def resolve(votes, available, quantity, reserve):
    counts = {action: sum(v["action"] == action for v in votes) for action in ["sell", "retain"]}
    chosen = "sell" if counts["sell"] > counts["retain"] else "retain"
    if chosen == "sell" and available - quantity < reserve:
        return "retain", "reserve invariant"
    return chosen, "retain on tie" if counts["sell"] == counts["retain"] else "majority"


print("judge :", judge.model_dump())
print("tie rule :", resolve(tied, FACTS["stock"], 2, 4))
print("policy override :", resolve([tied[0], tied[0]], FACTS["stock"], 4, 4))

judge : {'action': 'sell', 'reason': 'The proposal is to sell four units, matching the reserve of four; two of the six units would remain in stock.'}
tie rule : ('retain', 'retain on tie')
policy override : ('retain', 'reserve invariant')


Votes are proposals. They do not authorise a sale. Shared prompts and evidence can produce correlated mistakes. The judge's reason stays visible, and code rejects a reserve violation even when the majority favours it.

## Things to try

Optional : we can change the part, the context boundary or the policy and inspect the result.

In [6]:
finish_metrics("demo06")

metrics : {'model': 'openai/gpt-6-luna', 'rps': 0.4, 'calls': 4, 'input_tokens': 564, 'output_tokens': 141, 'seconds': 11.4}
